In [1]:
import pandas as pd
from scipy.io import loadmat
import pandas as pd
import re
import networkx as nx
import numpy as np
from collections import OrderedDict
from utils.prompting import *
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datasets import concatenate_datasets, load_dataset
from datasets import Dataset, DatasetDict
import ast

# Read Data

In [2]:
ground_truth_df = pd.read_pickle("../data/test/test.pkl")
ground_truth_df = ground_truth_df[['category', 'product_name', 'user_id', 'matched_kps']]
ground_truth_df = ground_truth_df.rename(columns={'matched_kps': 'key_point_given'})

In [3]:
ground_truth_df

,category,product_name,user_id,key_point_given
0,Beauty,Head & Shoulders Normal Hair Shampoo,3680,[Head & Shoulders can help with eczema for som...
1,Beauty,Gillette Mach 3 Razor,5000858,[Changing blades on the Mach 3 is quick and si...
2,Beauty,Pitrok,5296801,"[PitRok does not cause skin irritation, even f..."
3,Beauty,Gillette Mach 3 Razor,5050855,[Replacement blades for the Mach 3 are relativ...
4,Beauty,Timotei Golden Highlights Camomile Shampoo,5332164,"[It leaves hair feeling clean, soft, shiny, an..."
...,...,...,...,...
95,Travel,Dollar Rent A Car Worldwide,5297771,[Dollar Car Rental offers consistently lower p...
96,Travel,Amsterdam (Netherlands),5202501,[Accommodation ranges from budget hostels to l...
97,Travel,Leicester in General,5020891,[Leicester city centre offers a wide variety o...
98,Travel,Milan in general,5091015,[Buses in Milan are modern and comfortable but...


In [4]:
root_path = f"../output/stage_2_rl_inference/summary_kp_extraction"
temp_df = pd.read_pickle(root_path + "/1/1_done.pkl")
temp_df = temp_df.rename(columns={'voter_full': 'user_id'})
temp_df.shape

(96, 9)

In [5]:
temp_df['claim_split_predicted'] = temp_df['claim_split_predicted'].apply(lambda x: x.strip("```json").strip("\n"))
temp_df['claim_split_predicted'] = temp_df['claim_split_predicted'].apply(lambda x: ast.literal_eval(x))

In [6]:
temp_df = temp_df.rename(columns={'claim_split_predicted': 'key_point'})

In [7]:
claim_split_predicted = temp_df.merge(ground_truth_df)

In [8]:
claim_split_predicted

,key_point,personalized_summaries,category,product_name,user_id,product_reviews,hist_vote_written,filtered_hist_vote_written,my_category,key_point_given
0,[Head & Shoulders Normal Hair Shampoo is effec...,Here is a personalized summary of product A (H...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,[ Valium is something that probably everyon...,[ i love alberto balsams esspecially the su...,[ i love alberto balsams esspecially the su...,1,[Head & Shoulders can help with eczema for som...
1,[The Gillette Mach 3 Razor provides a close an...,Based on the user profile and the helpful key ...,Beauty,Gillette Mach 3 Razor,5000858,[ ~ ~ OK girls. This opinion is really for ...,[ When I had my red low lights done a coupl...,[ I bought these for going to festivals in ...,1,[Changing blades on the Mach 3 is quick and si...
2,[Pitrok is a natural deodorant free from artif...,Based on the helpful key points and the user p...,Beauty,Pitrok,5296801,[ this is the natural option to fight per...,[ 8776; Dandruff - the evil flaky whit...,[ 02 is I think one of the best innovations...,1,"[PitRok does not cause skin irritation, even f..."
3,[The Gillette Mach 3 Razor delivers a close an...,Here is a personalized summary of product A (G...,Beauty,Gillette Mach 3 Razor,5050855,[ ~ ~ OK girls. This opinion is really for ...,[ They are the 23.30.From five days I don t...,[ i love alberto balsams esspecially the su...,1,[Replacement blades for the Mach 3 are relativ...
4,"[The shampoo is gentle and effective., It is a...",Based on the user profile and the helpful key ...,Beauty,Timotei Golden Highlights Camomile Shampoo,5332164,[ Searching the freebie sites I found sache...,"[ Has a lovely light smell, really pretty a...","[ Has a lovely light smell, really pretty a...",1,"[It leaves hair feeling clean, soft, shiny, an..."
...,...,...,...,...,...,...,...,...,...,...
91,[Dollar Rent A Car Worldwide is a reliable and...,Based on the helpful key points and user 111's...,Travel,Dollar Rent A Car Worldwide,5297771,[ AH THE DREAM.- The thrill of the open ro...,[ AH THE DREAM.- The thrill of the open ro...,[ AH THE DREAM.- The thrill of the open ro...,1,[Dollar Car Rental offers consistently lower p...
92,[Amsterdam is steeped in tradition and is very...,Here is a personalized summary of product A (A...,Travel,Amsterdam (Netherlands),5202501,[ A week off of University and online trav...,[ Air Canada isn t that bad of an airline. ...,[ Air Canada isn t that bad of an airline. ...,1,[Accommodation ranges from budget hostels to l...
93,[Leicester offers a diverse range of shopping ...,Here is a personalized summary of product A (L...,Travel,Leicester in General,5020891,"[ Having lived here for 25 years now, this ...",[ Visitors to London who get the London Pas...,[ Visitors to London who get the London Pas...,1,[Leicester city centre offers a wide variety o...
94,[Milan features stunning architecture that imp...,Milan is a city that will leave you in awe. Fr...,Travel,Milan in general,5091015,[ Milculo is a swear word I was born near M...,[ Milan undoubtedly has some of the finest ...,[ Milan undoubtedly has some of the finest ...,1,[Buses in Milan are modern and comfortable but...


In [9]:
merged_df = claim_split_predicted.explode(['key_point']).explode(['key_point_given'])

In [10]:
merged_df

,key_point,personalized_summaries,category,product_name,user_id,product_reviews,hist_vote_written,filtered_hist_vote_written,my_category,key_point_given
0,Head & Shoulders Normal Hair Shampoo is effect...,Here is a personalized summary of product A (H...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,[ Valium is something that probably everyon...,[ i love alberto balsams esspecially the su...,[ i love alberto balsams esspecially the su...,1,Head & Shoulders can help with eczema for some...
0,Head & Shoulders Normal Hair Shampoo is effect...,Here is a personalized summary of product A (H...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,[ Valium is something that probably everyon...,[ i love alberto balsams esspecially the su...,[ i love alberto balsams esspecially the su...,1,Head & Shoulders does not permanently eradicat...
0,Head & Shoulders Normal Hair Shampoo is effect...,Here is a personalized summary of product A (H...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,[ Valium is something that probably everyon...,[ i love alberto balsams esspecially the su...,[ i love alberto balsams esspecially the su...,1,"The active ingredient, Pyrithione Zinc, is eff..."
0,Head & Shoulders Normal Hair Shampoo is effect...,Here is a personalized summary of product A (H...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,[ Valium is something that probably everyon...,[ i love alberto balsams esspecially the su...,[ i love alberto balsams esspecially the su...,1,Head & Shoulders is available in a range of fo...
0,"The shampoo leaves hair soft, clean, and shiny.",Here is a personalized summary of product A (H...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,[ Valium is something that probably everyon...,[ i love alberto balsams esspecially the su...,[ i love alberto balsams esspecially the su...,1,Head & Shoulders can help with eczema for some...
...,...,...,...,...,...,...,...,...,...,...
95,Visiting Budapest leaves you with unforgettabl...,Here is a personalized summary of Budapest tai...,Travel,Budapest (Hungary),5116809,[ With Easyjet now offering budget flights ...,"[ The Lenin Mausoleum, a red and black gran...",[],1,The Grand Palace overlooks the Danube and cont...
95,Visiting Budapest leaves you with unforgettabl...,Here is a personalized summary of Budapest tai...,Travel,Budapest (Hungary),5116809,[ With Easyjet now offering budget flights ...,"[ The Lenin Mausoleum, a red and black gran...",[],1,The Parliament buildings are magnificent and t...
95,Visiting Budapest leaves you with unforgettabl...,Here is a personalized summary of Budapest tai...,Travel,Budapest (Hungary),5116809,[ With Easyjet now offering budget flights ...,"[ The Lenin Mausoleum, a red and black gran...",[],1,"Travel within Budapest is extremely cheap, but..."
95,Visiting Budapest leaves you with unforgettabl...,Here is a personalized summary of Budapest tai...,Travel,Budapest (Hungary),5116809,[ With Easyjet now offering budget flights ...,"[ The Lenin Mausoleum, a red and black gran...",[],1,Vegetarian food is available and surprisingly ...


In [11]:
merged_df['key_point_given'] = merged_df['key_point_given'].apply(lambda x: x['key_point'] if type(x) == dict and 'key_point' in x else x)

In [12]:
merged_df = merged_df[['category', 'product_name', 'user_id', 'key_point', 'key_point_given']]

# Generated/Reference KP Matching

## Setup

In [13]:
from openai import OpenAI
client = OpenAI(
    api_key = "<YOUR-API-KEY"
)
model="gpt-4o-mini"

In [14]:
def get_completion(prompt, model=model):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=2000,
        temperature=0, # this is the degree of randomness of the model's output
    )
    return response.choices[0].message.content

In [15]:
base_prompt = get_prompt("shkp_kp_matching")

In [16]:
row = merged_df.iloc[11]
row

category                                                      Beauty
product_name                    Head & Shoulders Normal Hair Shampoo
user_id                                                         3680
key_point                      It works well for dry and curly hair.
key_point_given    Head & Shoulders is available in a range of fo...
Name: 0, dtype: object

In [17]:
prompt = base_prompt.format(product_name = row['product_name'], key_point = row['key_point'], key_point_given = row['key_point_given'])

In [18]:
response = get_completion(prompt, model)

In [19]:
print(response)

The generated key point states that the shampoo "works well for dry and curly hair," while the reference helpful key point mentions that "Head & Shoulders is available in a range of formulas for different hair types." 

Both key points discuss the effectiveness of Head & Shoulders shampoo in relation to different hair types. The generated key point specifically highlights its suitability for dry and curly hair, which falls under the broader category of different hair types mentioned in the reference key point. 

Therefore, the generated key point does match the reference helpful key point as they both relate to the product's effectiveness for various hair types.

**Answer: Yes**


## Inference

In [20]:
def get_kp_matching(product_name, key_point, key_point_given):
    random.seed(42)
    prompt = base_prompt.format(product_name = product_name,
                                key_point = key_point, 
                                key_point_given = key_point_given)
    retries = 2
    while retries > 0:
        try:
            response = get_completion(prompt, model)
            return response
        except Exception as e:
            if e:
                if "exceeded your current quota" in str(e).lower():
                    raise e
                print(e)
                print('Timeout error, retrying...')
                retries -= 1
                if "limit reached for" in str(e).lower():
                    time.sleep(30)
                else:
                    time.sleep(5)
            else:
                raise e

    print('API is not responding, moving on...')
    return None

In [21]:
def prompted_kp_matching_eval(root_path, domain, domain_df, save_step=100):
    src_path = f"{root_path}/{domain}"
    Path(src_path).mkdir(parents=True, exist_ok=True)
    kp_matching_responses = []

    file_names = listdir(src_path)
    postfix = [re.split("[_.]", name)[1]
               for name in listdir(src_path)
               ]
    start = 0
    if 'done' in postfix:
        print(domain, ": ", "Loaded saved file. Done")
        new_domain_df = pd.read_pickle(f"{src_path}/{domain}_done.pkl")
        return new_domain_df
    elif len(postfix) > 0:
        last_index = max([int(idx) for idx in postfix if idx != 'done'])
        last_domain_df = pd.read_pickle(f"{src_path}/{domain}_{last_index}.pkl")
        kp_matching_responses = last_domain_df['kp_matching_responses'].tolist()
        start = last_index
        print(domain, "Loaded saved file. Continuing")
    else:
        print(domain, "Start new process.")

    for i, (_, row) in tqdm(enumerate(domain_df.iterrows()), total=domain_df.shape[0]):
        if i < start:
            continue
        
        product_name = row['product_name']
        key_point = row['key_point']
        key_point_given = row['key_point_given']

        response = get_kp_matching(product_name, key_point, key_point_given)
        kp_matching_responses += [response]
        time.sleep(0.1)
        
        if (i + 1) % save_step == 0:
            save_df = domain_df.iloc[:i + 1]
            save_df.insert(0, 'kp_matching_responses', kp_matching_responses)
            save_df.to_pickle(f"{src_path}/{domain}_{i + 1}.pkl")

    new_domain_df = domain_df.iloc[:i + 1]
    new_domain_df.insert(0, 'kp_matching_responses', kp_matching_responses)
    new_domain_df.to_pickle(f"{src_path}/{domain}_done.pkl")
    return new_domain_df

In [22]:
merged_df['my_category'] = 1

/tmp/ipykernel_3957/2183625030.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_df['my_category'] = 1


In [23]:
merged_df

,category,product_name,user_id,key_point,key_point_given,my_category
0,Beauty,Head & Shoulders Normal Hair Shampoo,3680,Head & Shoulders Normal Hair Shampoo is effect...,Head & Shoulders can help with eczema for some...,1
0,Beauty,Head & Shoulders Normal Hair Shampoo,3680,Head & Shoulders Normal Hair Shampoo is effect...,Head & Shoulders does not permanently eradicat...,1
0,Beauty,Head & Shoulders Normal Hair Shampoo,3680,Head & Shoulders Normal Hair Shampoo is effect...,"The active ingredient, Pyrithione Zinc, is eff...",1
0,Beauty,Head & Shoulders Normal Hair Shampoo,3680,Head & Shoulders Normal Hair Shampoo is effect...,Head & Shoulders is available in a range of fo...,1
0,Beauty,Head & Shoulders Normal Hair Shampoo,3680,"The shampoo leaves hair soft, clean, and shiny.",Head & Shoulders can help with eczema for some...,1
...,...,...,...,...,...,...
95,Travel,Budapest (Hungary),5116809,Visiting Budapest leaves you with unforgettabl...,The Grand Palace overlooks the Danube and cont...,1
95,Travel,Budapest (Hungary),5116809,Visiting Budapest leaves you with unforgettabl...,The Parliament buildings are magnificent and t...,1
95,Travel,Budapest (Hungary),5116809,Visiting Budapest leaves you with unforgettabl...,"Travel within Budapest is extremely cheap, but...",1
95,Travel,Budapest (Hungary),5116809,Visiting Budapest leaves you with unforgettabl...,Vegetarian food is available and surprisingly ...,1


In [24]:
root_path = f"../output/stage_2_rl_inference/shkp_matching"
inputs = [(root_path,
           domain,
           merged_df[merged_df['my_category'] == domain].reset_index(drop=True)
           )
          for domain in merged_df['my_category'].unique()]

In [25]:
num_workers = 1

In [26]:
from datasets import concatenate_datasets, load_dataset
from datasets import Dataset, DatasetDict
import pandas as pd
import numpy as np
import torch
import os
import ast
import time
from multiprocessing import Pool
from pathlib import Path
from os import listdir
from tqdm import tqdm
import random
import re
import math
# import spacy
# pd.set_option('display.max_colwidth', None)

In [63]:
start_time = time.time()
with Pool(num_workers) as processor:
    data = processor.starmap(prompted_kp_matching_eval, inputs)
print("TIME ELAPSED", time.time() - start_time)

1 Loaded saved file. Continuing


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 13131/13131 [6:35:18<00:00,  1.81s/it]


TIME ELAPSED 23718.889353990555


In [27]:
processed_merged_df = pd.read_pickle(root_path + "/1/1_done.pkl")

In [28]:
processed_merged_df.iloc[-13]['kp_matching_responses']

'The generated key point "Visiting Budapest leaves you with unforgettable memories" does not match the reference helpful key point "Dreher lager is tasty and inexpensive in traditional Budapest pubs." \n\nWhile both points relate to the experience of being in Budapest, the generated key point is more general and focuses on the overall experience of visiting the city, whereas the reference key point specifically discusses a particular product (Dreher lager) and its affordability in a specific context (traditional pubs). They do not implicitly discuss the same issues or viewpoints, as one is about the overall experience and the other is about a specific aspect of that experience. \n\nTherefore, the answer is **No**, the generated key point does not match the reference helpful key point.'

In [29]:
processed_merged_df

,kp_matching_responses,category,product_name,voter_full,key_point,key_point_given,my_category
0,The generated key point discusses the effectiv...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,Head & Shoulders Normal Hair Shampoo is effect...,Head & Shoulders can help with eczema for some...,1
1,The generated key point states that Head & Sho...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,Head & Shoulders Normal Hair Shampoo is effect...,Head & Shoulders does not permanently eradicat...,1
2,"Yes, the generated key point matches the refer...",Beauty,Head & Shoulders Normal Hair Shampoo,3680,Head & Shoulders Normal Hair Shampoo is effect...,"The active ingredient, Pyrithione Zinc, is eff...",1
3,The generated key point discusses the effectiv...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,Head & Shoulders Normal Hair Shampoo is effect...,Head & Shoulders is available in a range of fo...,1
4,"No, the generated key point does not match the...",Beauty,Head & Shoulders Normal Hair Shampoo,3680,"The shampoo leaves hair soft, clean, and shiny.",Head & Shoulders can help with eczema for some...,1
...,...,...,...,...,...,...,...
13126,"The generated key point (""Visiting Budapest le...",Travel,Budapest (Hungary),5116809,Visiting Budapest leaves you with unforgettabl...,The Grand Palace overlooks the Danube and cont...,1
13127,"The generated key point (""Visiting Budapest le...",Travel,Budapest (Hungary),5116809,Visiting Budapest leaves you with unforgettabl...,The Parliament buildings are magnificent and t...,1
13128,"The generated key point (""Visiting Budapest le...",Travel,Budapest (Hungary),5116809,Visiting Budapest leaves you with unforgettabl...,"Travel within Budapest is extremely cheap, but...",1
13129,"The generated key point (""Visiting Budapest le...",Travel,Budapest (Hungary),5116809,Visiting Budapest leaves you with unforgettabl...,Vegetarian food is available and surprisingly ...,1


# Calculate SHKP

In [30]:
def label_helpfulness(response):
    if "yes" in response.lower():
        return True
    elif "does not" in response.lower() or "no" in response.lower():
        return False

In [31]:
processed_merged_df['kp_matching_label'] = processed_merged_df['kp_matching_responses'].apply(label_helpfulness)

In [32]:
processed_merged_df

,kp_matching_responses,category,product_name,voter_full,key_point,key_point_given,my_category,kp_matching_label
0,The generated key point discusses the effectiv...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,Head & Shoulders Normal Hair Shampoo is effect...,Head & Shoulders can help with eczema for some...,1,True
1,The generated key point states that Head & Sho...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,Head & Shoulders Normal Hair Shampoo is effect...,Head & Shoulders does not permanently eradicat...,1,True
2,"Yes, the generated key point matches the refer...",Beauty,Head & Shoulders Normal Hair Shampoo,3680,Head & Shoulders Normal Hair Shampoo is effect...,"The active ingredient, Pyrithione Zinc, is eff...",1,True
3,The generated key point discusses the effectiv...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,Head & Shoulders Normal Hair Shampoo is effect...,Head & Shoulders is available in a range of fo...,1,False
4,"No, the generated key point does not match the...",Beauty,Head & Shoulders Normal Hair Shampoo,3680,"The shampoo leaves hair soft, clean, and shiny.",Head & Shoulders can help with eczema for some...,1,False
...,...,...,...,...,...,...,...,...
13126,"The generated key point (""Visiting Budapest le...",Travel,Budapest (Hungary),5116809,Visiting Budapest leaves you with unforgettabl...,The Grand Palace overlooks the Danube and cont...,1,False
13127,"The generated key point (""Visiting Budapest le...",Travel,Budapest (Hungary),5116809,Visiting Budapest leaves you with unforgettabl...,The Parliament buildings are magnificent and t...,1,False
13128,"The generated key point (""Visiting Budapest le...",Travel,Budapest (Hungary),5116809,Visiting Budapest leaves you with unforgettabl...,"Travel within Budapest is extremely cheap, but...",1,False
13129,"The generated key point (""Visiting Budapest le...",Travel,Budapest (Hungary),5116809,Visiting Budapest leaves you with unforgettabl...,Vegetarian food is available and surprisingly ...,1,False


## Precision

In [33]:
precision_df = processed_merged_df.groupby(['category', 'product_name', 'voter_full', 'key_point']).agg({
    'kp_matching_label': (lambda x: x.tolist())
}).reset_index()

In [34]:
precision_df['matching_count'] = precision_df['kp_matching_label'].apply(lambda x: len([label for label in x if label == True]))

In [35]:
precision_df

,category,product_name,voter_full,key_point,kp_matching_label,matching_count
0,Beauty,Gillette Mach 3 Razor,5000858,The Gillette Mach 3 Razor provides a close and...,"[False, False, False, True, True, False, True,...",10
1,Beauty,Gillette Mach 3 Razor,5000858,The Mach 3 Razor is effective at providing a c...,"[True, False, False, True, True, False, True, ...",10
2,Beauty,Gillette Mach 3 Razor,5000858,The razor has a comfortable grip and a simple ...,"[False, False, False, True, True, False, False...",9
3,Beauty,Gillette Mach 3 Razor,5000858,The razor is a good value for the price.,"[False, False, True, False, False, False, True...",8
4,Beauty,Gillette Mach 3 Razor,5000858,The razor is a great choice for those with sen...,"[False, False, False, True, True, False, True,...",7
...,...,...,...,...,...,...
825,Travel,Milan in general,5091015,Milan provides an authentic Italian dining exp...,"[False, False, False, False, False, False, Fal...",2
826,Travel,Milan in general,5091015,Milan's vibrant shopping scene is a highlight ...,"[False, False, False, False, False, False, Fal...",3
827,Travel,Milan in general,5091015,The city has a strong sense of community that ...,"[False, True, False, False, False, False, Fals...",2
828,Travel,Milan in general,5091015,The city is a paradise for fashion lovers.,"[False, True, False, False, False, False, Fals...",5


In [36]:
precision_df = precision_df.groupby(['category', 'product_name', 'voter_full']).agg({
    'matching_count': (lambda x: x.tolist())
}).reset_index()

In [37]:
precision_df['matching_kps'] = precision_df['matching_count'].apply(lambda x: len([num_match for num_match in x if num_match > 1]))

In [38]:
precision_df['helpful_kp_proportion'] = precision_df.apply(lambda row: row['matching_kps'] / len(row['matching_count']), axis=1)

In [39]:
precision_df

,category,product_name,voter_full,matching_count,matching_kps,helpful_kp_proportion
0,Beauty,Gillette Mach 3 Razor,5000858,"[10, 10, 9, 8, 7, 8, 4, 9]",8,1.000000
1,Beauty,Gillette Mach 3 Razor,5050855,"[1, 2, 7, 6, 5, 2, 3, 6]",7,0.875000
2,Beauty,Head & Shoulders Normal Hair Shampoo,3680,"[3, 1, 0, 2, 1, 0, 0, 1, 0, 0]",2,0.200000
3,Beauty,Pitrok,5296801,"[7, 6, 2, 5, 1, 2, 7]",6,0.857143
4,Beauty,Timotei Golden Highlights Camomile Shampoo,5332164,"[4, 6, 1, 1, 3, 1, 3, 5, 2, 3]",7,0.700000
...,...,...,...,...,...,...
91,Travel,Amsterdam (Netherlands),5202501,"[7, 11, 7, 4, 5, 5, 14]",7,1.000000
92,Travel,Budapest (Hungary),5116809,"[1, 3, 7, 9, 2, 4, 4, 6, 3, 2, 5]",10,0.909091
93,Travel,Dollar Rent A Car Worldwide,5297771,"[5, 8, 10, 2, 6, 4, 2, 4, 4, 4, 3]",11,1.000000
94,Travel,Leicester in General,5020891,"[3, 2, 1, 4, 8, 1, 1, 4, 2]",6,0.666667


In [40]:
precision = precision_df['helpful_kp_proportion'].mean()
precision

0.8388666021478522

## Recall

In [41]:
recall_df = processed_merged_df.groupby(['category', 'product_name', 'voter_full', 'key_point_given']).agg({
    'kp_matching_label': (lambda x: x.tolist())
}).reset_index()

In [42]:
recall_df['matching_count'] = recall_df['kp_matching_label'].apply(lambda x: len([label for label in x if label == True]))

In [43]:
recall_df = recall_df.groupby(['category', 'product_name', 'voter_full']).agg({
    'matching_count': (lambda x: x.tolist())
}).reset_index()

In [44]:
recall_df['matching_kps'] = recall_df['matching_count'].apply(lambda x: len([num_match for num_match in x if num_match > 0]))

In [45]:
recall_df['helpful_kp_proportion'] = recall_df.apply(lambda row: row['matching_kps'] / len(row['matching_count']), axis=1)

In [46]:
recall_df

,category,product_name,voter_full,matching_count,matching_kps,helpful_kp_proportion
0,Beauty,Gillette Mach 3 Razor,5000858,"[3, 1, 7, 0, 3, 5, 8, 6, 6, 5, 4, 2, 5, 4, 6]",14,0.933333
1,Beauty,Gillette Mach 3 Razor,5050855,"[3, 2, 5, 5, 4, 6, 0, 2, 5]",8,0.888889
2,Beauty,Head & Shoulders Normal Hair Shampoo,3680,"[1, 3, 2, 2]",4,1.000000
3,Beauty,Pitrok,5296801,"[5, 4, 3, 3, 0, 6, 0, 1, 1, 4, 2, 0, 1, 0]",10,0.714286
4,Beauty,Timotei Golden Highlights Camomile Shampoo,5332164,"[6, 3, 4, 0, 1, 2, 4, 2, 3, 2, 2]",10,0.909091
...,...,...,...,...,...,...
91,Travel,Amsterdam (Netherlands),5202501,"[1, 2, 5, 5, 3, 3, 1, 0, 3, 1, 1, 1, 0, 3, 2, ...",21,0.913043
92,Travel,Budapest (Hungary),5116809,"[5, 1, 6, 6, 6, 1, 1, 0, 1, 0, 1, 5, 1, 2, 1, ...",19,0.863636
93,Travel,Dollar Rent A Car Worldwide,5297771,"[6, 3, 0, 9, 3, 5, 5, 5, 2, 2, 6, 6, 0]",11,0.846154
94,Travel,Leicester in General,5020891,"[2, 1, 1, 2, 5, 0, 1, 1, 1, 3, 2, 1, 1, 2, 3, 0]",14,0.875000


In [47]:
recall = recall_df['helpful_kp_proportion'].mean()
recall

0.8309663018169554

## F1

In [48]:
f1 = 2 * (precision * recall) / (precision + recall)
f1

0.8348977630629406